# SDXL Optimized API — Deployment Demo

This notebook demonstrates the LitServe API serving the optimized SDXL model.

**Preset options:**
| Preset | Steps | Expected Latency | Speedup | Use Case |
|--------|-------|-----------------|---------|----------|
| `speed` | 4 | ~0.85s | 6.7× | Real-time/interactive |
| `balanced` | 8 | ~1.0s | 5.5× | Production APIs |
| `quality` | 50 | ~3.4s | 1.7× | Quality-critical |

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content')
REPO_URL = 'https://github.com/rajat116/sdxl-optimization.git'
if not os.path.exists('/content/sdxl-optimization/src'):
    !git clone {REPO_URL}
os.chdir('/content/sdxl-optimization')
!pip install -q -e '.[dev]' && pip install -q DeepCache
print('✅ Ready')

In [ ]:
# Start LitServe server
import subprocess, requests, time, base64, json
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

PRESET = 'speed'  # Change to 'balanced' or 'quality' to compare

!kill $(lsof -t -i:8000) 2>/dev/null || true
time.sleep(2)

proc = subprocess.Popen(
    ['python', 'server/serve.py', '--preset', PRESET, '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print(f'Starting LitServe with \'{PRESET}\' preset...')
print('Waiting for model to load (3-8 min on T4)...\n')

server_ready = False
for i in range(480):
    if proc.poll() is not None:
        print(f'❌ Server crashed'); break
    try:
        r = requests.post('http://localhost:8000/predict',
                          json={'prompt': 'test', 'height': 64, 'width': 64}, timeout=3)
        if r.status_code == 200:
            print(f'✅ Server ready after {i+1}s!')
            server_ready = True; break
    except: pass
    time.sleep(1)
    if i % 60 == 59: print(f'  Still loading... ({i+1}s)')

In [ ]:
# === API Demo: curl-equivalent call ===
print('━' * 60)
print('  API ENDPOINT TEST')
print('━' * 60)

payload = {
    'prompt': 'a photo of an astronaut riding a horse on mars, cinematic lighting',
    'seed': 42,
    'height': 1024,
    'width': 1024,
}

print(f'\ncurl -X POST http://localhost:8000/predict \\')
print(f'  -H "Content-Type: application/json" \\')
print(f'  -d \'{json.dumps(payload)}\'\n')

response = requests.post('http://localhost:8000/predict', json=payload, timeout=120)
data = response.json()

print(f'Status:  {response.status_code}')
print(f'Latency: {data["latency_s"]}s')
print(f'Preset:  {data["preset"]} — {data["preset_info"]["description"]}')
print(f'Speedup: {data["preset_info"]["expected_speedup"]} vs baseline')

img = Image.open(BytesIO(base64.b64decode(data['image_base64'])))
display(img)

In [ ]:
# === Batch test: 4 prompts through the API ===
prompts = [
    'a photo of an astronaut riding a horse on mars, cinematic lighting',
    'a beautiful sunset over a calm ocean with sailboats, oil painting',
    'a corgi wearing a top hat and monocle, portrait photography',
    'a futuristic cityscape at night with neon lights, cyberpunk',
]

imgs, lats = [], []
for i, p in enumerate(prompts):
    r = requests.post('http://localhost:8000/predict',
                      json={'prompt': p, 'seed': 42}, timeout=120)
    d = r.json()
    imgs.append(Image.open(BytesIO(base64.b64decode(d['image_base64']))))
    lats.append(d['latency_s'])
    print(f'  [{i+1}/4] {d["latency_s"]}s')

avg = sum(lats)/len(lats)
print(f'\n📊 Avg: {avg:.2f}s | Throughput: {60/avg:.0f} img/min | Speedup: {5.66/avg:.1f}×')

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, im, p, l in zip(axes, imgs, prompts, lats):
    ax.imshow(im); ax.set_title(f'{l}s\n{p[:35]}...', fontsize=9); ax.axis('off')
plt.suptitle(f'LitServe API — \'{PRESET}\' preset | {avg:.2f}s/img | {60/avg:.0f} img/min',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('results/api_batch_demo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cleanup
proc.terminate(); proc.wait()
print('✅ Server stopped.')